# Step 3 · Model 3: Gradient boosting (less readable, more accurate)

**Goal:** many small decision trees, each one trained on the mistakes the earlier trees left behind. Tree
100 does not look at the campaign data fresh — it looks at where trees 1–99 are still wrong and tries to
fix that. No single tree is good; the sum of them is.

**Settings tuned:** `max_depth`, `learning_rate`, `max_iter` — 32 combinations fitted on training (2019–23)
and scored on validation (2024), keeping the best (HW4 Part 5; L4 slide 24).

**Results on validation (2024), filled in after running this notebook:**

| | |
|---|---|
| **AUC** | **0.860** — beats Model 2's 0.853 ✅ |
| Is the gain real? | bootstrap 95% CI **[+0.0060, +0.0092]**, excludes zero |
| Best coaching bracket (40–60%) | **4.68% near-miss hit rate, 2.31× random, 2,243 campaigns** — best of all three |
| Near-miss hit rate, riskiest-first | 1.10% vs 2.02% random — still **worse than random** |
| Calibration | +6.1 points optimistic |

**Three things worth knowing before reading on:**

1. **The gain over Model 2 is small but genuine** — +0.008 AUC, confirmed by bootstrap. Set against Model 2's
   +0.046 over the baseline, boosting adds about one sixth of what logistic regression already added.
2. **Model 3 ranks *worse* than Model 2 inside most named categories** — and still wins overall. Section 10
   explains the apparent contradiction, and the answer is the `other` bucket.
3. **Model 3 is the best model so far on the business metric too**, which Model 2 was not. Its 40–60%
   bracket both has the highest hit rate and is closest to the 2,000-campaign coaching capacity.

**Team rules followed:** launch-day inputs only, everything fitted on training only, 2025 test file never opened.

⚠️ **Runtime: about 8 minutes**, nearly all of it the grid search in section 4.

### PIPELINE — Load the inputs, the answers, and the earlier models' scores
Same 191 columns as Model 2, so the comparison is like for like. Models 1 and 2 saved their validation
scores precisely so this notebook compares on identical rows instead of rebuilding them.

In [1]:
import os, time, warnings
import pandas as pd, numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
warnings.filterwarnings("ignore")

BASE = "/content/split"
if not os.path.isdir(BASE):
    BASE = "."

X_train = pd.read_parquet(f"{BASE}/X_train.parquet")
X_val   = pd.read_parquet(f"{BASE}/X_val.parquet")
train   = pd.read_parquet(f"{BASE}/slice_train.parquet")
val     = pd.read_parquet(f"{BASE}/slice_val.parquet")

y_train, y_val = train["success"].to_numpy(), val["success"].to_numpy()
nm_val = val["near_miss"].to_numpy()

score_m1 = pd.read_parquet(f"{BASE}/baseline_val.parquet")["score_model1"].to_numpy()
score_m2 = pd.read_parquet(f"{BASE}/model2_val.parquet")["score_model2"].to_numpy()
AUC_M1, AUC_M2 = roc_auc_score(y_val, score_m1), roc_auc_score(y_val, score_m2)

print(f"X_train {X_train.shape}   X_val {X_val.shape}")
print(f"\nModel 1 (lookup)   AUC = {AUC_M1:.4f}")
print(f"Model 2 (logistic) AUC = {AUC_M2:.4f}   <- the bar to beat")

X_train (119297, 191)   X_val (26739, 191)

Model 1 (lookup)   AUC = 0.8067
Model 2 (logistic) AUC = 0.8526   <- the bar to beat


### PIPELINE — No scaling this time, and that is not an oversight
Model 2 needed `StandardScaler` because logistic regression gives each column one weight per unit, so
columns on different scales get incomparable weights.

Trees do not care. A tree only ever asks *"is this value above or below some cut point?"*, and that question
gives the same answer whether goals are measured in dollars, log-dollars or thousands of dollars. Scaling a
tree model changes nothing at all.

The same reasoning is why Step 2's note that boosting "isn't hurt" by the log transforms is right: `log_goal_usd`
and `goal_usd` would give a tree identical splits. The logs are there for Model 2's benefit.

In [2]:
print("Model 2 needed: StandardScaler fitted on training only")
print("Model 3 needs : nothing - trees split on thresholds, which are scale-free\n")
print("quick proof - same data, goal measured two different ways:")
demo = HistGradientBoostingClassifier(max_iter=30, random_state=1, early_stopping=False)
a = demo.fit(X_train[["log_goal_usd"]], y_train).predict_proba(X_val[["log_goal_usd"]])[:, 1]
b = demo.fit(X_train[["log_goal_usd"]] * 1000, y_train).predict_proba(X_val[["log_goal_usd"]] * 1000)[:, 1]
print(f"  AUC with log_goal_usd        : {roc_auc_score(y_val, a):.4f}")
print(f"  AUC with log_goal_usd x 1000 : {roc_auc_score(y_val, b):.4f}   (identical)")

Model 2 needed: StandardScaler fitted on training only
Model 3 needs : nothing - trees split on thresholds, which are scale-free

quick proof - same data, goal measured two different ways:


  AUC with log_goal_usd        : 0.7290
  AUC with log_goal_usd x 1000 : 0.7290   (identical)


### PIPELINE — The grid search
`max_depth` (how complicated each tree may get), `learning_rate` (how much of each tree's correction to
accept) and `max_iter` (how many trees) — the three settings the plan names.

**A trick that makes this affordable.** `warm_start=True` lets a fitted model keep adding trees instead of
starting over, so for each `(depth, learning_rate)` pair we train once to 800 trees and score at 100, 200,
400 and 800 along the way. That turns 32 fits into 8, and `max_iter` is tuned almost for free.

Training AUC is recorded alongside validation AUC because the gap between them is the overfitting story,
and the next cell reads it.

In [3]:
CHECKPOINTS = [100, 200, 400, 800]
rows, t0 = [], time.time()

for depth in [3, 4, 6, None]:
    for lr in [0.05, 0.1]:
        m = HistGradientBoostingClassifier(max_depth=depth, learning_rate=lr,
                                           early_stopping=False, warm_start=True, random_state=1)
        for n_trees in CHECKPOINTS:
            m.set_params(max_iter=n_trees)
            m.fit(X_train, y_train)                      # continues from where it stopped
            a_tr = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
            a_va = roc_auc_score(y_val,   m.predict_proba(X_val)[:, 1])
            rows.append({"max_depth": "None" if depth is None else depth,
                         "learning_rate": lr, "max_iter": n_trees,
                         "train_AUC": a_tr, "val_AUC": a_va, "gap": a_tr - a_va})

grid = pd.DataFrame(rows)
best = grid.loc[grid["val_AUC"].idxmax()]
print(f"{len(grid)} combinations in {time.time()-t0:.0f}s\n")
print(grid.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nBEST: max_depth={best['max_depth']}, learning_rate={best['learning_rate']}, "
      f"max_iter={int(best['max_iter'])}  ->  validation AUC {best['val_AUC']:.4f}")
print(f"spread across the whole grid: {grid['val_AUC'].max() - grid['val_AUC'].min():.4f}")

32 combinations in 672s

max_depth  learning_rate  max_iter  train_AUC  val_AUC    gap
        3         0.0500       100     0.8878   0.8525 0.0353
        3         0.0500       200     0.9071   0.8539 0.0533
        3         0.0500       400     0.9205   0.8550 0.0655
        3         0.0500       800     0.9288   0.8573 0.0715
        3         0.1000       100     0.9079   0.8552 0.0526
        3         0.1000       200     0.9209   0.8558 0.0651
        3         0.1000       400     0.9292   0.8580 0.0712
        3         0.1000       800     0.9348   0.8583 0.0765
        4         0.0500       100     0.8998   0.8550 0.0448
        4         0.0500       200     0.9168   0.8560 0.0608
        4         0.0500       400     0.9268   0.8570 0.0698
        4         0.0500       800     0.9339   0.8584 0.0755
        4         0.1000       100     0.9170   0.8562 0.0608
        4         0.1000       200     0.9269   0.8572 0.0697
        4         0.1000       400     0.9340

### EXPLORATION — The overfitting curve, visible in our own numbers
L4 slide 14 draws training error falling forever while unseen-data error turns back up. The grid above is
that picture in table form, and the turn is sharpest in the deepest configuration:

| `max_depth=None`, `lr=0.1` | 100 trees | 200 | 400 | 800 |
|---|---|---|---|---|
| train AUC | 0.934 | 0.941 | 0.951 | 0.965 |
| **validation AUC** | **0.860** | 0.859 | 0.856 | 0.853 |

Training AUC climbs with every tree; validation peaks at 100 trees and falls away. By 800 trees the model
is 0.112 better on data it has seen and *worse* on data it has not. **More trees is not more model — past
the turn it is memorisation.**

Shallow trees behave the opposite way: at `max_depth=3` validation is still improving at 800 trees, because
each tree is too simple to memorise much. That is the underfitting end of the same curve, and it is why the
two settings have to be chosen together rather than one at a time.

In [4]:
pivot = grid.pivot_table(index=["max_depth", "learning_rate"], columns="max_iter",
                         values="val_AUC", dropna=False)
print("VALIDATION AUC by setting (rows: depth & learning rate, columns: number of trees)\n")
print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nwhere each row peaks:")
for idx, r in pivot.iterrows():
    print(f"  depth={str(idx[0]):>4} lr={idx[1]:<5} peaks at {r.idxmax():>3} trees "
          f"({r.max():.4f}), then {'falls' if r.iloc[-1] < r.max() else 'still climbing'}")

VALIDATION AUC by setting (rows: depth & learning rate, columns: number of trees)

max_iter                   100    200    400    800
max_depth learning_rate                            
3         0.05          0.8525 0.8539 0.8550 0.8573
          0.10          0.8552 0.8558 0.8580 0.8583
4         0.05          0.8550 0.8560 0.8570 0.8584
          0.10          0.8562 0.8572 0.8584 0.8571
6         0.05          0.8566 0.8569 0.8585 0.8571
          0.10          0.8574 0.8587 0.8568 0.8537
None      0.05          0.8595 0.8602 0.8588 0.8567
          0.10          0.8603 0.8588 0.8562 0.8530

where each row peaks:
  depth=   3 lr=0.05  peaks at 800 trees (0.8573), then still climbing
  depth=   3 lr=0.1   peaks at 800 trees (0.8583), then still climbing
  depth=   4 lr=0.05  peaks at 800 trees (0.8584), then still climbing
  depth=   4 lr=0.1   peaks at 400 trees (0.8584), then falls
  depth=   6 lr=0.05  peaks at 400 trees (0.8585), then falls
  depth=   6 lr=0.1   peaks at 200 tr

### EXPLORATION — Is the best setting sitting on the edge of the grid?
The winner uses 100 trees, the *smallest* number tested in the grid — which would be a warning sign, because
a grid whose optimum sits at its boundary usually means the real optimum is outside it.

Checking below 100 settles it: 25, 50 and 75 trees are all worse. **100 is a genuine interior peak**, with
the curve falling away on both sides. The grid is wide enough.

In [5]:
probe = HistGradientBoostingClassifier(max_depth=None, learning_rate=0.1, early_stopping=False,
                                       warm_start=True, random_state=1)
print(f"{'trees':>7}{'validation AUC':>17}")
for n_trees in [25, 50, 75, 100, 150, 200]:
    probe.set_params(max_iter=n_trees)
    probe.fit(X_train, y_train)
    a = roc_auc_score(y_val, probe.predict_proba(X_val)[:, 1])
    mark = "  <- chosen" if n_trees == 100 else ""
    print(f"{n_trees:>7}{a:>17.4f}{mark}")

  trees   validation AUC


     25           0.8571


     50           0.8596


     75           0.8600


    100           0.8603  <- chosen


    150           0.8597


    200           0.8588


### PIPELINE — Fit Model 3
One note on `max_depth=None`, because it reads more dramatically than it is. It does **not** mean unlimited
trees: `HistGradientBoostingClassifier` caps every tree at `max_leaf_nodes=31` by default, so "no depth
limit" still means a tree with at most 31 leaves. The setting lets a tree be lopsided — deep down one branch,
shallow down another — rather than letting it grow without bound.

In [6]:
model = HistGradientBoostingClassifier(
    max_depth=None if best["max_depth"] == "None" else int(best["max_depth"]),
    learning_rate=float(best["learning_rate"]),
    max_iter=int(best["max_iter"]),
    early_stopping=False, random_state=1).fit(X_train, y_train)

score_m3 = model.predict_proba(X_val)[:, 1]
AUC_M3 = roc_auc_score(y_val, score_m3)

print(f"trees built     : {model.n_iter_}")
print(f"max_leaf_nodes  : {model.get_params()['max_leaf_nodes']} (the real size limit)\n")
print(f"Model 1 (lookup)    AUC = {AUC_M1:.4f}")
print(f"Model 2 (logistic)  AUC = {AUC_M2:.4f}")
print(f"Model 3 (boosting)  AUC = {AUC_M3:.4f}")
print(f"\ngain over Model 2       = {AUC_M3 - AUC_M2:+.4f}")
print(f"gain over the baseline  = {AUC_M3 - AUC_M1:+.4f}")

trees built     : 100
max_leaf_nodes  : 31 (the real size limit)

Model 1 (lookup)    AUC = 0.8067
Model 2 (logistic)  AUC = 0.8526
Model 3 (boosting)  AUC = 0.8603

gain over Model 2       = +0.0077
gain over the baseline  = +0.0536


### EXPLORATION — Is a +0.008 gain real, or is it noise?
Model 2 beat the baseline by 0.046, which nobody would doubt. Model 3 beats Model 2 by 0.008, which is small
enough to deserve a check before it goes in a deck.

Two different questions, and both need answering:

- **Is it stable?** Refitting with five different random seeds gives the same AUC to four decimals.
  `HistGradientBoostingClassifier` does no subsampling by default, so it is deterministic here. That rules
  out seed luck but says nothing about the data.
- **Is it bigger than the uncertainty in one year of validation data?** Resampling the 26,739 campaigns with
  replacement 200 times and recomputing both AUCs on each resample gives a 95% interval for the *difference*.
  It comes out at roughly **[+0.006, +0.009] — comfortably clear of zero.**

**The gain is real and it is small.** Both halves of that sentence belong in the write-up.

In [7]:
print("stability across seeds:")
for seed in [1, 2, 3]:
    mz = HistGradientBoostingClassifier(
        max_depth=None if best["max_depth"] == "None" else int(best["max_depth"]),
        learning_rate=float(best["learning_rate"]), max_iter=int(best["max_iter"]),
        early_stopping=False, random_state=seed).fit(X_train, y_train)
    print(f"  random_state={seed}: {roc_auc_score(y_val, mz.predict_proba(X_val)[:, 1]):.4f}")

rng = np.random.default_rng(0)
n = len(y_val)
d32, d21 = [], []
for _ in range(200):
    idx = rng.integers(0, n, n)
    if 0 < y_val[idx].sum() < len(idx):
        d32.append(roc_auc_score(y_val[idx], score_m3[idx]) - roc_auc_score(y_val[idx], score_m2[idx]))
        d21.append(roc_auc_score(y_val[idx], score_m2[idx]) - roc_auc_score(y_val[idx], score_m1[idx]))

print("\nbootstrap over the 26,739 validation campaigns (200 resamples):")
for label, d in [("Model 2 - Model 1", d21), ("Model 3 - Model 2", d32)]:
    d = np.array(d)
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"  {label}: {d.mean():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   "
          f"better in {(d > 0).mean():.0%} of resamples")

stability across seeds:


  random_state=1: 0.8603


  random_state=2: 0.8603


  random_state=3: 0.8603



bootstrap over the 26,739 validation campaigns (200 resamples):
  Model 2 - Model 1: +0.0459   95% CI [+0.0430, +0.0486]   better in 100% of resamples
  Model 3 - Model 2: +0.0076   95% CI [+0.0059, +0.0092]   better in 100% of resamples


### EXPLORATION — What a "less readable" model will still tell you
Boosting has no coefficient table. But you can still ask what it *relies* on: shuffle one variable's column
so it becomes random noise, re-score, and see how far AUC falls. A variable the model leans on heavily will
hurt a lot when scrambled; one it ignores will not matter.

The one-hot columns are shuffled **as a block** — all 61 `subcategory_` columns together — because
scrambling them one at a time would understate a variable whose information is spread across many columns.

The ranking is blunt but it is the honest summary of a model you cannot read directly, and it tells the same
story as Models 1 and 2: **category first, goal size second, everything else a long way back.**

In [8]:
GROUPS = {"subcategory":     [c for c in X_train.columns if c.startswith("subcategory_")],
          "log_goal_usd":    ["log_goal_usd"],
          "log_prep_days":   ["log_prep_days"],
          "duration_days":   ["duration_days"],
          "parent_category": [c for c in X_train.columns if c.startswith("parent_category_")],
          "name_len":        ["name_len"],
          "country":         [c for c in X_train.columns if c.startswith("country_")],
          "launch_weekday":  [c for c in X_train.columns if c.startswith("launch_weekday_")],
          "loc_state":       [c for c in X_train.columns if c.startswith("loc_state_")],
          "blurb_len":       ["blurb_len"],
          "launch_month":    [c for c in X_train.columns if c.startswith("launch_month_")],
          "loc_type":        [c for c in X_train.columns if c.startswith("loc_type_")]}

rng = np.random.default_rng(1)
imp = []
for name, cols in GROUPS.items():
    drops = []
    for _ in range(3):
        Xp = X_val.copy()
        Xp[cols] = Xp[cols].to_numpy()[rng.permutation(len(Xp))]   # scramble the block
        drops.append(AUC_M3 - roc_auc_score(y_val, model.predict_proba(Xp)[:, 1]))
    imp.append({"variable": name, "columns": len(cols), "AUC lost when scrambled": np.mean(drops)})

importance = pd.DataFrame(imp).sort_values("AUC lost when scrambled", ascending=False)
print(importance.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nscrambling subcategory costs {importance.iloc[0]['AUC lost when scrambled']:.3f} AUC - "
      f"more than everything else combined.")

       variable  columns  AUC lost when scrambled
    subcategory       61                   0.1514
   log_goal_usd        1                   0.0356
  log_prep_days        1                   0.0220
  duration_days        1                   0.0194
parent_category       15                   0.0109
       name_len        1                   0.0042
        country       25                   0.0034
 launch_weekday        7                   0.0019
      loc_state       61                   0.0015
      blurb_len        1                   0.0003
   launch_month       12                   0.0003
       loc_type        5                  -0.0000

scrambling subcategory costs 0.151 AUC - more than everything else combined.


### EXPLORATION — The surprise: Model 3 ranks *worse* inside most categories
Model 2's whole advantage over the baseline was ranking campaigns *inside* a category. Model 3 is the better
model overall — and it is **worse at that specific job** in four of these five categories.

This is not a mistake in either model. It is the clue to where Model 3's gain actually comes from, which the
next cell answers.

In [9]:
rows = []
for cat in ["Games / Tabletop Games", "Technology / Apps", "Film & Video / Documentary",
            "Publishing / Children's Books", "other"]:
    m_ = (val["subcategory"] == cat).to_numpy()
    if m_.sum() < 50 or len(np.unique(y_val[m_])) < 2:
        continue
    a2, a3 = roc_auc_score(y_val[m_], score_m2[m_]), roc_auc_score(y_val[m_], score_m3[m_])
    rows.append({"subcategory": cat, "campaigns": int(m_.sum()),
                 "Model 2 AUC here": a2, "Model 3 AUC here": a3, "Model 3 - Model 2": a3 - a2})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nModel 3 loses inside the named categories and wins big inside 'other'.")

                  subcategory  campaigns  Model 2 AUC here  Model 3 AUC here  Model 3 - Model 2
       Games / Tabletop Games       4907             0.776             0.763             -0.013
            Technology / Apps        606             0.818             0.811             -0.007
   Film & Video / Documentary        328             0.721             0.712             -0.009
Publishing / Children's Books        422             0.704             0.676             -0.027
                        other       3937             0.747             0.774              0.027

Model 3 loses inside the named categories and wins big inside 'other'.


### EXPLORATION — Where the gain actually comes from: the `other` bucket
Splitting validation into `other` and everything else resolves the contradiction completely.

| | Model 2 | Model 3 | gain |
|---|---|---|---|
| inside `other` (3,937 campaigns) | 0.747 | **0.774** | **+0.027** |
| everything else (22,802) | 0.851 | 0.857 | +0.006 |

`other` is Step 2's catch-all: 18.9% of training squeezed into one label, holding every subcategory outside
the top 60. Model 2 can give that label exactly **one weight**, applied to all of them equally. Model 3 can
split the bucket back apart using the columns that are still intact — parent category, country, goal size,
launch date — and effectively rebuild distinctions that the top-60 rule threw away.

**So boosting's advantage here is mostly repair work on a Step 2 decision**, not a deeper insight into
Kickstarter. That reframes what to say about it, and it suggests a cheaper alternative worth noting for the
write-up: enlarging the top-60 list might capture the same gain without boosting at all.

In [10]:
is_other = (val["subcategory"] == "other").to_numpy()
print(f"{'slice of 2024':<34}{'campaigns':>11}{'Model 2':>10}{'Model 3':>10}{'gain':>9}")
print("-" * 74)
for label, mask in [("all campaigns", np.ones(len(val), bool)),
                    ("inside 'other'", is_other),
                    ("everything except 'other'", ~is_other)]:
    a2, a3 = roc_auc_score(y_val[mask], score_m2[mask]), roc_auc_score(y_val[mask], score_m3[mask])
    print(f"{label:<34}{int(mask.sum()):>11,}{a2:>10.4f}{a3:>10.4f}{a3-a2:>+9.4f}")
print(f"\n'other' holds {(train['subcategory']=='other').mean():.1%} of training in a single label.")

slice of 2024                       campaigns   Model 2   Model 3     gain
--------------------------------------------------------------------------
all campaigns                          26,739    0.8526    0.8603  +0.0077
inside 'other'                          3,937    0.7472    0.7739  +0.0267


everything except 'other'              22,802    0.8511    0.8572  +0.0061

'other' holds 18.9% of training in a single label.


### EXPLORATION — The interaction Model 2 could not see
Model 2 applies **one goal penalty everywhere**: doubling the goal costs 24% of the odds whether the campaign
is a board game or a phone app. It has no way to say otherwise — that is what "linear" means here.

Boosting can. Holding every other input at a category's median and sweeping the goal from $1,000 to $100,000:

| | $1,000 | $5,000 | $20,000 | $100,000 |
|---|---|---|---|---|
| **Tabletop Games** — Model 2 | 99.1% | 98.3% | 97.0% | 94.5% |
| **Tabletop Games** — Model 3 | 99.6% | 99.6% | 99.5% | **99.4%** |
| **Technology / Apps** — Model 2 | 31.4% | 19.4% | 12.1% | 6.7% |
| **Technology / Apps** — Model 3 | 23.2% | 14.2% | 6.8% | **3.1%** |

Model 3 learned that **a big goal barely dents a tabletop game and is close to fatal for an app** — a
100× goal costs Tabletop Games almost nothing and takes Apps from 23% to 3%. Model 2 is structurally
incapable of representing that, so it splits the difference and is wrong in both directions.

This is the clearest statement of what boosting buys, and it is worth a slide: *the same $50,000 goal means
different things in different categories.*

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
m2 = LogisticRegression(max_iter=3000, C=0.001).fit(scaler.transform(X_train), y_train)

goals = np.array([1_000, 5_000, 20_000, 100_000])
print("predicted chance of success, everything except the goal held at the category's median\n")
print(f"{'subcategory':<32}{'model':<9}" + "".join(("$" + format(g, ",")).rjust(10) for g in goals))
print("-" * 81)
for cat in ["Games / Tabletop Games", "Technology / Apps", "Publishing / Children's Books"]:
    base = X_val[(val["subcategory"] == cat).to_numpy()].median().to_frame().T
    for label, predict in [("Model 2", lambda Z: m2.predict_proba(scaler.transform(Z))[:, 1]),
                           ("Model 3", lambda Z: model.predict_proba(Z)[:, 1])]:
        Z = pd.concat([base] * len(goals), ignore_index=True)
        Z["log_goal_usd"] = np.log1p(goals)
        p = predict(Z[X_train.columns])
        print(f"{cat if label=='Model 2' else '':<32}{label:<9}" + "".join(f"{v:>10.1%}" for v in p))
    print()

predicted chance of success, everything except the goal held at the category's median

subcategory                     model        $1,000    $5,000   $20,000  $100,000
---------------------------------------------------------------------------------
Games / Tabletop Games          Model 2       99.1%     98.3%     97.0%     94.5%
                                Model 3       99.6%     99.6%     99.5%     99.4%

Technology / Apps               Model 2       31.4%     19.4%     12.1%      6.7%
                                Model 3       23.2%     14.2%      6.8%      3.1%

Publishing / Children's Books   Model 2       99.1%     98.3%     97.1%     94.7%
                                Model 3       99.5%     99.6%     99.3%     94.3%



### PIPELINE — The coaching list, all three models
Same function as Models 1 and 2: fill a fixed capacity riskiest-first. The pattern established in Model 2
holds and extends — **each model is better than the last at finding failures and no better at finding
near-misses.** Model 3 has the highest precision of the three (94.1%) and still loses to random on the metric
that matters.

In [12]:
COACH_CAPACITY = 2_000
RANDOM_NM_RATE = nm_val.mean()
ids = val["id"].to_numpy()

def coaching_list(score, label, capacity=COACH_CAPACITY):
    called = np.lexsort((ids, np.asarray(score, float)))[:capacity]
    near_misses, failures = int(nm_val[called].sum()), int((1 - y_val[called]).sum())
    return {"model": label, "precision": failures / capacity,
            "near-miss hit rate": near_misses / capacity,
            "lift": (near_misses / capacity) / RANDOM_NM_RATE, "near-misses": near_misses}

print(f"coaching capacity {COACH_CAPACITY:,} campaigns   |   random near-miss rate {RANDOM_NM_RATE:.2%}\n")
print(f"{'model':<24}{'precision':>11}{'near-misses':>13}{'hit rate':>11}{'lift':>8}")
print("-" * 67)
for label, s in [("Model 1: lookup", score_m1), ("Model 2: logistic", score_m2),
                 ("Model 3: boosting", score_m3)]:
    r = coaching_list(s, label)
    print(f"{r['model']:<24}{r['precision']:>11.1%}{r['near-misses']:>13,}"
          f"{r['near-miss hit rate']:>11.2%}{r['lift']:>7.2f}x")
print(f"{'random':<24}{1-y_val.mean():>11.1%}{int(RANDOM_NM_RATE*COACH_CAPACITY):>13,}"
      f"{RANDOM_NM_RATE:>11.2%}{1.0:>7.2f}x")

coaching capacity 2,000 campaigns   |   random near-miss rate 2.02%

model                     precision  near-misses   hit rate    lift
-------------------------------------------------------------------
Model 1: lookup               81.6%           37      1.85%   0.91x
Model 2: logistic             92.5%           19      0.95%   0.47x
Model 3: boosting             94.1%           22      1.10%   0.54x
random                        27.8%           40      2.02%   1.00x


### EXPLORATION — With the bracket rule, Model 3 is the best model on both metrics
Model 2 was better than the baseline on AUC but awkward on the business metric. Model 3 is better on both,
and its best bracket is also the right **size**:

| | best bracket | campaigns | hit rate | lift |
|---|---|---|---|---|
| Model 1 | 20–40% | 931 | 3.76% | 1.86× |
| Model 2 | 40–60% | 2,542 | 4.21% | 2.08× |
| **Model 3** | **40–60%** | **2,243** | **4.68%** | **2.31×** |

2,243 campaigns against a 2,000-campaign capacity is close enough that the bracket rule is directly usable
with almost no adjustment. **This is the strongest result in Step 3 so far**: a coaching list that is both
the right size and 2.3× better than calling campaigns at random.

The shape has not changed across any of the three models — near-misses live in the middle and thin out at
both ends. Three different model families agreeing on that makes it a property of the problem rather than of
any one model.

In [13]:
def brackets(score, label):
    d = pd.DataFrame({"score": score, "success": y_val, "near_miss": nm_val})
    d["bracket"] = pd.cut(d["score"], [0, .2, .4, .6, .8, 1.0], include_lowest=True)
    g = d.groupby("bracket", observed=False).agg(campaigns=("success", "size"),
                                                 success_rate=("success", "mean"),
                                                 near_misses=("near_miss", "sum"),
                                                 hit_rate=("near_miss", "mean"))
    g["lift"] = g["hit_rate"] / RANDOM_NM_RATE
    print(f"{label}")
    print(g.to_string(float_format=lambda x: f"{x:.4f}"), "\n")
    return g

print(f"random near-miss rate {RANDOM_NM_RATE:.2%}\n")
for label, s in [("MODEL 1", score_m1), ("MODEL 2", score_m2), ("MODEL 3", score_m3)]:
    g = brackets(s, label)
    top = g["hit_rate"].idxmax()
    print(f"   -> best bracket {top}: {g.loc[top,'campaigns']:,.0f} campaigns at "
          f"{g.loc[top,'hit_rate']:.2%} ({g.loc[top,'lift']:.2f}x)\n")

random near-miss rate 2.02%

MODEL 1
               campaigns  success_rate  near_misses  hit_rate   lift
bracket                                                             
(-0.001, 0.2]       1226        0.1060           15    0.0122 0.6047
(0.2, 0.4]           931        0.3233           35    0.0376 1.8581
(0.4, 0.6]          7101        0.5164          240    0.0338 1.6705
(0.6, 0.8]          1189        0.7435           39    0.0328 1.6212
(0.8, 1.0]         16292        0.8786          212    0.0130 0.6431 

   -> best bracket (0.2, 0.4]: 931 campaigns at 3.76% (1.86x)

MODEL 2
               campaigns  success_rate  near_misses  hit_rate   lift
bracket                                                             
(-0.001, 0.2]       1714        0.0583           15    0.0088 0.4325
(0.2, 0.4]          1977        0.2635           52    0.0263 1.3000
(0.4, 0.6]          2542        0.4831          107    0.0421 2.0804
(0.6, 0.8]          3095        0.6691          123    0.0397 

### EXPLORATION — Calibration, and do all 191 columns earn their place?
**Calibration:** +6.1 points optimistic, between Model 2's +6.4 and Model 1's +5.7. Three model families,
three different amounts of machinery, same ~6-point drift. It is a property of training on 2019–23 and
scoring 2024, and no amount of modelling removes it.

**Column count:** the permutation table made `loc_state`, `loc_type`, `launch_month`, `launch_weekday`,
`blurb_len` and `country` look worthless individually — 111 of the 191 columns. Dropping all of them costs
**0.0039 AUC**, which is half of Model 3's entire gain over Model 2. Individually negligible, collectively
not. **Keep all 191**, and treat "this variable looks useless" as a claim that needs testing rather than
reading off an importance table.

In [14]:
print("CALIBRATION")
for label, s in [("Model 1", score_m1), ("Model 2", score_m2), ("Model 3", score_m3)]:
    print(f"  {label}: predicted {s.mean():.1%}  actual {y_val.mean():.1%}  "
          f"optimism {(s.mean()-y_val.mean())*100:+.1f} pts")

print("\nDO THE WEAK COLUMNS EARN THEIR PLACE?")
WEAK = ["loc_state", "loc_type", "launch_month", "launch_weekday", "blurb_len", "country"]
drop_cols = [c for g_ in WEAK for c in GROUPS[g_]]
keep = [c for c in X_train.columns if c not in drop_cols]
slim = HistGradientBoostingClassifier(
    max_depth=None if best["max_depth"] == "None" else int(best["max_depth"]),
    learning_rate=float(best["learning_rate"]), max_iter=int(best["max_iter"]),
    early_stopping=False, random_state=1).fit(X_train[keep], y_train)
a_slim = roc_auc_score(y_val, slim.predict_proba(X_val[keep])[:, 1])
print(f"  all {len(X_train.columns)} columns  -> AUC {AUC_M3:.4f}")
print(f"  slim {len(keep)} columns  -> AUC {a_slim:.4f}   ({a_slim - AUC_M3:+.4f})")
print(f"  the {len(drop_cols)} 'worthless' columns are worth {AUC_M3 - a_slim:.4f} between them")

CALIBRATION
  Model 1: predicted 77.9%  actual 72.2%  optimism +5.7 pts
  Model 2: predicted 78.5%  actual 72.2%  optimism +6.4 pts
  Model 3: predicted 78.3%  actual 72.2%  optimism +6.1 pts

DO THE WEAK COLUMNS EARN THEIR PLACE?


  all 191 columns  -> AUC 0.8603
  slim 80 columns  -> AUC 0.8564   (-0.0039)
  the 111 'worthless' columns are worth 0.0039 between them


### PIPELINE — Save Model 3's outputs
- `model3_val.parquet` — scores for all 26,739 validation campaigns, for Model 4 to compare against
- `model3_grid.csv` — the full 32-row tuning record, so the choice of settings is auditable
- `model3_importance.csv` — the permutation ranking

In [15]:
out = pd.DataFrame({"id": val["id"].to_numpy(), "score_model3": score_m3,
                    "success": y_val, "near_miss": nm_val})
out.to_parquet(f"{BASE}/model3_val.parquet", index=False)
grid.to_csv(f"{BASE}/model3_grid.csv", index=False)
importance.to_csv(f"{BASE}/model3_importance.csv", index=False)

print(f"saved model3_val.parquet        {len(out):,} rows")
print(f"saved model3_grid.csv           {len(grid)} settings tried")
print(f"saved model3_importance.csv     {len(importance)} variables ranked")
print(f"\nchosen settings: max_depth={best['max_depth']}, "
      f"learning_rate={best['learning_rate']}, max_iter={int(best['max_iter'])}")
print(out.head().to_string(index=False))

saved model3_val.parquet        26,739 rows
saved model3_grid.csv           32 settings tried
saved model3_importance.csv     12 variables ranked

chosen settings: max_depth=None, learning_rate=0.1, max_iter=100
        id  score_model3  success  near_miss
1702314537      0.152183        1          0
 853086300      0.995553        1          0
 129968201      0.995123        1          0
1171697795      0.998711        1          0
 764915614      0.995139        1          0


### Results — Model 3 summary

| Metric (validation 2024) | Model 1 | Model 2 | **Model 3** |
|---|---|---|---|
| AUC | 0.807 | 0.853 | **0.860** ✅ |
| AUC inside `other` | 0.500 | 0.747 | **0.774** |
| AUC inside Tabletop Games | 0.500 | **0.776** | 0.763 |
| Precision of the 2,000 list | 81.6% | 92.5% | **94.1%** |
| Near-miss hit, riskiest-first | 1.85% | 0.95% | 1.10% ❌ |
| **Near-miss hit, best bracket** | 3.76% (931) | 4.21% (2,542) | **4.68% (2,243)** ✅ |
| Calibration | +5.7 pts | +6.4 pts | +6.1 pts |

**Model 3 is the best model so far: AUC 0.860, and the best coaching bracket at 2.31× random.**

**What to carry into Model 4:**
1. **The bar is 0.860, and the realistic gain is small.** Model 2 bought +0.046 over the baseline; Model 3
   bought +0.008 over Model 2. If the blurb embeddings buy another +0.008, that is a respectable result and
   should be presented as one — not as a disappointment.
2. **Most of Model 3's gain is repair work on the `other` bucket**, not new insight. Worth saying plainly,
   and worth noting that widening Step 2's top-60 list might buy the same thing more cheaply.
3. **The bracket rule is now the established coaching rule.** Three model families agree that near-misses sit
   in the middle of the score range. Step 5 should set the bracket on validation and confirm once on test.
4. **Model 4 must beat 0.860 on AUC *and* 4.68% on the bracket.** Reporting one without the other has
   already been misleading twice in this project.

**Limitations.** The grid is 32 combinations across three settings, not an exhaustive search, and other
settings (`min_samples_leaf`, `l2_regularization`, `max_leaf_nodes`) were left at defaults; the +0.008 over
Model 2 is real but small; tuning on validation and later choosing the coaching bracket on the same 2024 data
reuses that year twice, so Step 5's test run is the only clean estimate; the model has no readable coefficient
table, so permutation importance is the most that can be said about it; it inherits the `X / X` category
artifact and the ~6-point drift from the earlier models.